# 08. 실시간 시세 연동 & assets.csv 자동 업데이트

이 Notebook에서 하는 일:
1. assets.csv에서 ticker(종목코드)가 있는 자산 추출
2. FinanceDataReader / pykrx로 현재 시세 조회
3. current_value 자동 계산 (수량 × 현재가)
4. 현금성 자산(ticker 없음)은 수동 입력 유지
5. 업데이트된 assets.csv 저장

> **실행 순서**: 이 노트북 실행 후 → 01_data_input.ipynb 실행하면 최신 시세로 분석됩니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
import time

PROJECT_ROOT = Path.cwd()
assets_path  = PROJECT_ROOT / 'assets.csv'
sample_path  = PROJECT_ROOT / 'assets_sample.csv'

# assets.csv 로드
if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig', dtype={'ticker': str})
    print(f'assets.csv 로드 — {len(df)}개 자산')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig', dtype={'ticker': str})
    print(f'샘플 데이터 로드 — {len(df)}개 자산')

# ticker 정리 — float으로 읽힌 경우(예: '153130.0') 정수 문자열로 변환
def clean_ticker(x):
    s = str(x).strip()
    if s in ('', 'nan', 'NaN', 'None'):
        return ''
    try:
        return str(int(float(s)))
    except:
        return s

df['ticker']        = df['ticker'].apply(clean_ticker)
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)
df['quantity']      = pd.to_numeric(df['quantity'],      errors='coerce').fillna(0)
df['unit_price']    = pd.to_numeric(df['unit_price'],    errors='coerce').fillna(0)

# ticker 있는 것 / 없는 것 분리
has_ticker = df['ticker'] != ''
print(f'\n시세 조회 대상: {has_ticker.sum()}개')
print(f'수동 유지 대상: {(~has_ticker).sum()}개 (현금성 자산)')
print()
print(df[has_ticker][['asset_name','ticker','quantity','current_value']].to_string(index=False))

assets.csv 로드 — 32개 자산

시세 조회 대상: 8개
수동 유지 대상: 24개 (현금성 자산)

     asset_name ticker  quantity  current_value
 KODEX 미국배당다우존스 489250    1882.0              0
      KODEX 200  69500     961.0              0
 KODEX 미국S&P500 379800     895.0              0
     KODEX 고배당주 279530    2359.0              0
TIGER 리츠부동산 인프라 329200    8224.0              0
          캠트로닉스  89010     100.0              0
             루닛 328130      12.0              0
           기가비스 420770      80.0              0


In [ ]:
# ── 만료 자산 자동 감지 (maturity_date 기준) ────────────────
# maturity_date가 입력된 cash/bond/fund 중 만기일 < 오늘이면 is_active=0
from datetime import date as _date

EXPIRABLE_TYPES = ['cash', 'bond', 'fund']
today_d = _date.today()

# is_active / maturity_date 컬럼 없으면 추가
if 'is_active' not in df.columns:
    df['is_active'] = 1
if 'maturity_date' not in df.columns:
    df['maturity_date'] = ''

df['is_active']     = df['is_active'].fillna(1).astype(int)
df['maturity_date'] = df['maturity_date'].fillna('').astype(str).str.strip()

newly_expired = []
for i, row in df.iterrows():
    mat_str = str(row['maturity_date']).strip()
    if not mat_str or mat_str in ('nan', 'NaT', ''):
        continue
    if str(row['asset_type']) in EXPIRABLE_TYPES and int(row['is_active']) == 1:
        try:
            if _date.fromisoformat(mat_str) < today_d:
                df.at[i, 'is_active'] = 0
                newly_expired.append(row['asset_name'])
        except:
            pass

if newly_expired:
    print(f'⚠️  만기 도래 자산 {len(newly_expired)}개 → 비활성(is_active=0) 처리:')
    for n in newly_expired:
        print(f'   - {n}')
else:
    print('✅ 만료 자산 없음 (만기일 미기입 또는 모두 만기 전)')

active_count   = int((df['is_active'] == 1).sum())
inactive_count = int((df['is_active'] == 0).sum())
print(f'\n현재 활성: {active_count}개 / 비활성: {inactive_count}개')


## 1. 시세 조회 함수

In [2]:
import FinanceDataReader as fdr

def get_price_fdr(ticker):
    try:
        ticker = ticker.zfill(6)
        end    = date.today()
        start  = end - timedelta(days=10)
        df_p   = fdr.DataReader(ticker, start=str(start), end=str(end))
        if df_p.empty:
            return None, None
        return float(df_p['Close'].iloc[-1]), str(df_p.index[-1].date())
    except:
        return None, None

def get_price_pykrx(ticker):
    try:
        from pykrx import stock
        ticker = ticker.zfill(6)
        today  = date.today().strftime('%Y%m%d')
        start  = (date.today() - timedelta(days=10)).strftime('%Y%m%d')
        df_p   = stock.get_market_ohlcv_by_date(start, today, ticker)
        if df_p.empty:
            return None, None
        return float(df_p['종가'].iloc[-1]), str(df_p.index[-1].date())
    except:
        return None, None

def get_price(ticker):
    price, pdate = get_price_fdr(ticker)
    if price:
        return price, pdate, 'FDR'
    price, pdate = get_price_pykrx(ticker)
    if price:
        return price, pdate, 'pykrx'
    return None, None, 'failed'

print('시세 조회 함수 준비 완료')

시세 조회 함수 준비 완료


## 2. 실시간 시세 조회

In [3]:
print('=== 실시간 시세 조회 중 ===')
print()

df_updated = df.copy()
results    = []

for idx, row in df[has_ticker].iterrows():
    ticker     = row['ticker']
    asset_name = row['asset_name']
    quantity   = row['quantity']
    old_value  = row['current_value']

    price, price_date, source = get_price(ticker)
    time.sleep(0.3)

    if price and quantity > 0:
        new_value  = price * quantity
        change     = new_value - old_value
        change_pct = (change / old_value * 100) if old_value > 0 else 0
        df_updated.at[idx, 'unit_price']    = price
        df_updated.at[idx, 'current_value'] = new_value
        print(f'✅ [{ticker}] {asset_name}')
        print(f'   현재가: {price:>10,.0f}원  수량: {quantity:.0f}주')
        print(f'   평가액: {new_value:>12,.0f}원  (변동: {change:+,.0f}원 / {change_pct:+.1f}%)')
        print(f'   기준일: {price_date}  출처: {source}')
        results.append({'status': 'ok', 'name': asset_name, 'change': change})
    elif price and quantity == 0:
        print(f'⚠️  [{ticker}] {asset_name} — 수량 0, current_value 수동 유지')
        results.append({'status': 'manual', 'name': asset_name, 'change': 0})
    else:
        print(f'❌  [{ticker}] {asset_name} — 시세 조회 실패, 기존값 유지')
        results.append({'status': 'failed', 'name': asset_name, 'change': 0})
    print()

ok   = sum(1 for r in results if r['status'] == 'ok')
fail = sum(1 for r in results if r['status'] == 'failed')
print(f'조회 완료: {ok}개 성공 / {fail}개 실패')

=== 실시간 시세 조회 중 ===

✅ [489250] KODEX 미국배당다우존스
   현재가:     13,230원  수량: 1882주
   평가액:   24,898,860원  (변동: +24,898,860원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [69500] KODEX 200
   현재가:    113,340원  수량: 961주
   평가액:  108,919,740원  (변동: +108,919,740원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [379800] KODEX 미국S&P500
   현재가:     25,080원  수량: 895주
   평가액:   22,446,600원  (변동: +22,446,600원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [279530] KODEX 고배당주
   현재가:     17,575원  수량: 2359주
   평가액:   41,459,425원  (변동: +41,459,425원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [329200] TIGER 리츠부동산 인프라
   현재가:      4,180원  수량: 8224주
   평가액:   34,376,320원  (변동: +34,376,320원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [89010] 캠트로닉스
   현재가:     36,200원  수량: 100주
   평가액:    3,620,000원  (변동: +3,620,000원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [328130] 루닛
   현재가:     16,940원  수량: 12주
   평가액:      203,280원  (변동: +203,280원 / +0.0%)
   기준일: 2026-05-20  출처: FDR

✅ [420770] 기가비스
   현재가:    121,000원  수량: 80주
   평가액:    9,680,000원

## 3. 현금성 자산 수동 업데이트 (선택)

In [4]:
cash_df = df[~has_ticker][['account_name','asset_name','asset_type','current_value']]
print('=== 현금성 자산 (수동 업데이트) ===')
for idx, row in cash_df.iterrows():
    print(f'  [{idx}] {row["account_name"]} — {row["asset_name"]}: {row["current_value"]:,.0f}원')
print()
print('금액 변경 시 아래 딕셔너리에 index번호: 새금액 형식으로 입력 후 다음 셀 실행')

=== 현금성 자산 (수동 업데이트) ===
  [0] 연금저축 — 서울도시철도 24-06: 3,850,000원
  [1] 연금저축 — 키움글로벌얼터너티브증권(혼합-재)-C-P2: 433,400원
  [2] 연금저축 — NH 저축은행 정기예금DC: 40,938,608원
  [5] 연금저축 — 삼성증권 디폴트옵션 안정형 포트폴리오: 1,341,221원
  [6] 연금저축 — SBI저축은행 퇴직연금 IRP: 27,921,481원
  [7] 연금저축 — 삼성화재 이율보증형(3년): 22,605,072원
  [8] 연금저축 — 푸른저축은행 퇴직연금정기예금 IRP (1년): 39,487,027원
  [10] 연금저축 — DB저축은행 퇴직연금 정기예금 IRP(1년): 31,869,892원
  [13] 개인저축 — SAP SE: 53,618,224원
  [14] 개인저축 — 서울도시철도 24-06 -2: 33,980,558원
  [18] 개인저축 — 조비에비에이션: 8,758,291원
  [19] 개인저축 — T1.125 05/15/40: 2,157,609원
  [20] 개인저축 — T1.125 05/15/40: 19,699,912원
  [21] 개인연금 — 신한마음편한 TDF: 34,075,627원
  [22] 개인연금 — 한국투자크레딧포커스(채권): 29,656,135원
  [23] 개인연금 — SBI저축은행IRP: 21,320,275원
  [24] 개인연금 — 고려저축은행정기예금IRP(12M): 8,333,807원
  [25] 개인연금 — 푸른상호저축은행IRP(12): 53,391,149원
  [26] 개인연금 — 미래에셋 배분전략전갹TDF2050혼합자산자: 37,641,833원
  [27] 개인저축 — 쏠편한정기예금: 10,280,000원
  [28] 개인저축 — Kwak 예금: 77,353,000원
  [29] 개인저축 — 신탁형ISA: 63,723,716원
  [30] 개인저축 — 쏠편한정기예금: 13,390,000원
  [31] 개인저축 — MG 더뱅킹정기예금

In [5]:
# 변경할 현금성 자산 입력 (변동 없으면 빈 딕셔너리 그대로 실행)
# 예시: cash_updates = { 4: 32000000, 5: 6000000 }
cash_updates = {}

for idx, new_val in cash_updates.items():
    old_val = df_updated.at[idx, 'current_value']
    df_updated.at[idx, 'current_value'] = new_val
    print(f'업데이트: [{idx}] {df_updated.at[idx, "asset_name"]} {old_val:,.0f} → {new_val:,.0f}원')

if not cash_updates:
    print('현금성 자산 변동 없음 — 기존값 유지')

현금성 자산 변동 없음 — 기존값 유지


## 4. 변경 확인 및 저장

In [6]:
import shutil

old_total = df['current_value'].sum()
new_total = df_updated['current_value'].sum()
diff      = new_total - old_total

print('=== 업데이트 전후 비교 ===')
print(f'업데이트 전: {old_total:>15,.0f}원')
print(f'업데이트 후: {new_total:>15,.0f}원')
print(f'변동액:      {diff:>+15,.0f}원  ({diff/old_total*100:+.2f}%)')
print()

# 백업 후 저장
today_str  = date.today().strftime('%Y%m%d')
backup_dir = PROJECT_ROOT / 'backup'
backup_dir.mkdir(exist_ok=True)

if assets_path.exists():
    shutil.copy(assets_path, backup_dir / f'assets_{today_str}.csv')
    print(f'백업 저장: backup/assets_{today_str}.csv')

df_updated.to_csv(assets_path, index=False, encoding='utf-8-sig')
print(f'assets.csv 업데이트 완료!')
print()
print('다음: 01_data_input.ipynb 실행하여 최신 시세로 분석하세요.')

=== 업데이트 전후 비교 ===
업데이트 전:     645,826,837원
업데이트 후:     891,431,062원
변동액:         +245,604,225원  (+38.03%)

백업 저장: backup/assets_20260520.csv
assets.csv 업데이트 완료!

다음: 01_data_input.ipynb 실행하여 최신 시세로 분석하세요.


---
## 완료!
- ticker 있는 자산: 실시간 시세 자동 업데이트
- 현금성 자산: 수동 입력 유지
- 기존 파일: `backup/` 폴더에 날짜별 자동 보관

**다음 단계**: `01_data_input.ipynb` → `02~05` → `07_report.ipynb` 순서로 실행